In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = None

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ Successfully logged into Hugging Face Hub")
except Exception as e:
    print("⚠️ Could not retrieve HF_TOKEN from Kaggle secrets:", e)
    print("   Make sure you added the secret in Add-ons → Secrets")

✅ Successfully logged into Hugging Face Hub


In [3]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("⚠️ Running on CPU only")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [4]:
"""
Milestone 5 - Ensembling
Smart MCQ Solver
(fixed: see review notes at bottom of chat message)
"""

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)  # also seeds transformers/Trainer-internal RNG (dataloader shuffling etc.)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_path = f"{DATA_DIR}/train.csv"
test_path = f"{DATA_DIR}/test.csv"
if not os.path.exists(train_path) or not os.path.exists(test_path):
    raise FileNotFoundError(
        f"Expected train/test CSVs under {DATA_DIR}. "
        "Check DATA_DIR if you're not running inside the Kaggle environment."
    )

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

LABEL2ID = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
OPTION_COLS = ["A", "B", "C", "D", "E"]
NUM_LABELS = 5


def build_text(row, prefix=""):
    # str() guards against NaN/float option cells blowing up the tokenizer
    opts = "\n".join(f"{letter}) {str(row[letter])}" for letter in OPTION_COLS)
    return f"{prefix}{row['prompt']}\n{opts}"


# --- Validation split ---
# Shuffle before splitting so the held-out set isn't just "whatever rows happened
# to be first" in the CSV (which may be sorted by topic/difficulty/date and give
# a biased, unrepresentative validation score).
shuffled_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = shuffled_df.iloc[:100].reset_index(drop=True)
fit_df = shuffled_df.iloc[100:].reset_index(drop=True)
print(f"Training on {len(fit_df)} rows, holding out {len(val_df)} for validation")


class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.texts = [build_text(r) for _, r in df.iterrows()]
        self.labels = [LABEL2ID[a] for a in df["answer"]] if "answer" in df.columns else None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    return {"accuracy": float(acc)}


def make_training_args(output_dir):
    eval_key = "eval_strategy" if hasattr(TrainingArguments, "eval_strategy") else "evaluation_strategy"

    kwargs = {
        "output_dir": output_dir,
        "per_device_train_batch_size": 6 if torch.cuda.is_available() else 4,
        "per_device_eval_batch_size": 8 if torch.cuda.is_available() else 4,
        "num_train_epochs": 3,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "logging_steps": 50,
        eval_key: "epoch",
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "accuracy",  # explicit, not implicit eval_loss
        "greater_is_better": True,
        # fp16 disabled: DeBERTa-v3's disentangled attention layer has parameters
        # that surface as leaf FP16 tensors outside the normal fp32-master-weights
        # path, which crashes GradScaler.unscale_() with
        # "ValueError: Attempting to unscale FP16 gradients."
        # These are small models, so mixed precision isn't worth the fragility.
        "fp16": False,
        "report_to": "none",
        "seed": SEED,
    }
    return TrainingArguments(**kwargs)


def finetune(model_name, out_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_LABELS)
    model = model.to(device)

    train_ds = MCQDataset(fit_df, tokenizer)
    eval_ds = MCQDataset(val_df, tokenizer)

    training_args = make_training_args(out_dir)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print(f"\n🔥 Starting fine-tuning: {model_name}")
    trainer.train()
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)

    return model, tokenizer


# ====================== TRAINING ======================
deberta_model, deberta_tok = finetune("microsoft/deberta-v3-small", f"{OUTPUT_DIR}/deberta-ft")
roberta_model, roberta_tok = finetune("roberta-base", f"{OUTPUT_DIR}/roberta-ft")

deberta_model.eval()
roberta_model.eval()


@torch.no_grad()
def get_probs_batch(model, tokenizer, rows, prefix="", max_len=256, batch_size=16):
    """Batched inference. Returns an (N, NUM_LABELS) array of softmax probabilities."""
    all_probs = []
    texts = [build_text(r, prefix=prefix) for r in rows]
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        enc = tokenizer(
            batch_texts, truncation=True, max_length=max_len, padding=True, return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)


@torch.no_grad()
def get_probs(model, tokenizer, row, prefix=""):
    """Single-row convenience wrapper kept for the Q1-Q9 spot checks."""
    return get_probs_batch(model, tokenizer, [row], prefix=prefix, batch_size=1)[0]


def top3_string(probs):
    order = np.argsort(probs)[::-1][:3]
    return " ".join(ID2LABEL[i] for i in order)


def weighted_ensemble(p_deberta, p_roberta, w_deberta=0.65, w_roberta=0.35):
    assert abs((w_deberta + w_roberta) - 1.0) < 1e-6, "ensemble weights must sum to 1"
    return w_deberta * p_deberta + w_roberta * p_roberta


# Q1-Q4
sample_row = test_df.iloc[25]
p_deb_25 = get_probs(deberta_model, deberta_tok, sample_row)
p_rob_25 = get_probs(roberta_model, roberta_tok, sample_row)

print(f"\nQ1 -> DeBERTa top: {ID2LABEL[int(np.argmax(p_deb_25))]}, prob={p_deb_25.max():.4f}")
print(f"Q2 -> Simple avg top: {ID2LABEL[int(np.argmax((p_deb_25 + p_rob_25) / 2))]}")

weighted_25 = weighted_ensemble(p_deb_25, p_rob_25)
print(f"Q3 -> Weighted ensemble top: {ID2LABEL[int(np.argmax(weighted_25))]}")
print(f"Q4 -> Weighted top-3: {top3_string(weighted_25)}")

# Q5: Submission (batched, with per-batch error handling so one bad row
# doesn't kill the whole run)
test_rows = [row for _, row in test_df.iterrows()]
try:
    p_deb_all = get_probs_batch(deberta_model, deberta_tok, test_rows)
    p_rob_all = get_probs_batch(roberta_model, roberta_tok, test_rows)
except Exception as e:
    raise RuntimeError(f"Batched inference failed on test set: {e}") from e

w_all = weighted_ensemble(p_deb_all, p_rob_all)
preds = [
    {"id": row["id"], "prediction": top3_string(w_all[i])}
    for i, row in enumerate(test_rows)
]

submission = pd.DataFrame(preds)
submission_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)
print(f"\nQ5 -> {submission_path} written ({len(submission)} rows)")

# Q6: TTA
TTA_PREFIX = "Answer the following multiple-choice question carefully: "
tta_rows = test_rows[:50]
p_deb_notta = get_probs_batch(deberta_model, deberta_tok, tta_rows)
p_deb_tta = get_probs_batch(deberta_model, deberta_tok, tta_rows, prefix=TTA_PREFIX)
tta_changed = int(np.sum(np.argmax(p_deb_notta, axis=1) != np.argmax(p_deb_tta, axis=1)))
print(f"Q6 -> TTA changed top-1 on {tta_changed} rows")

# Q7-Q9
p_deb_100 = p_deb_all[:100]
p_rob_100 = p_rob_all[:100]
p_ens_100 = w_all[:100]

top1_diff = int(np.sum(np.argmax(p_deb_100, axis=1) != np.argmax(p_ens_100, axis=1)))
positive_gain = int(np.sum(p_ens_100.max(axis=1) > p_deb_100.max(axis=1)))
top3_diff = sum(
    top3_string(p_deb_100[i]) != top3_string(p_ens_100[i]) for i in range(len(p_deb_100))
)

print(f"Q7 -> Different top-1: {top1_diff}")
print(f"Q8 -> Positive confidence gain: {positive_gain}")
print(f"Q9 -> Different top-3: {top3_diff}")


# Q10: MAP@3
def map_at_3(true_labels, pred_strings):
    if not true_labels:
        return 0.0
    scores = [
        1.0 / (pred.split().index(true) + 1) if true in pred.split() else 0.0
        for true, pred in zip(true_labels, pred_strings)
    ]
    return sum(scores) / len(scores)


val_true = list(val_df["answer"])
val_rows = [row for _, row in val_df.iterrows()]
p_deb_val = get_probs_batch(deberta_model, deberta_tok, val_rows)
p_rob_val = get_probs_batch(roberta_model, roberta_tok, val_rows)
w_val = weighted_ensemble(p_deb_val, p_rob_val)
val_preds = [top3_string(w_val[i]) for i in range(len(val_rows))]

print(f"\nQ10 -> MAP@3 on held-out validation: {map_at_3(val_true, val_preds):.4f}")
print("\n✅ All done!")

Using device: cuda
Training on 1900 rows, holding out 100 for validation


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Q1 -> DeBERTa top: A, prob=nan
Q2 -> Simple avg top: A
Q3 -> Weighted ensemble top: A
Q4 -> Weighted top-3: E D C

Q5 -> /kaggle/working/submission.csv written (500 rows)
Q6 -> TTA changed top-1 on 0 rows
Q7 -> Different top-1: 0
Q8 -> Positive confidence gain: 0
Q9 -> Different top-3: 0

Q10 -> MAP@3 on held-out validation: 0.3383

✅ All done!
